# Adversarial Attacks and Identity Leakage in De-Identification Systems
## Empirical Study Implementation
**Paper:** Rosberg et al., IEEE TBIOM Vol. 8 No. 2, March 2026

**Datasets Used:**
- FaceForensics++ (original only) → adversarial attacks, FIVA de-id, Tables II/III/IV, Figure 5
- CelebA → Learned U-Net training + Robust ArcFace fine-tuning (replaces VGGFace2)
- LFW → Final benchmark Table V

**Hardware target:** RTX 3050 (limited VRAM) — mixed precision + gradient accumulation throughout

## Section 1: Install Dependencies

In [1]:
# Install all required packages
import subprocess, sys

packages = [
    'torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118',
    'insightface onnxruntime-gpu',
    'facenet-pytorch',
    'timm',
    'adaface-pytorch',  # may not exist on pypi; handled below
    'scikit-learn scikit-image',
    'matplotlib seaborn pandas',
    'Pillow tqdm',
    'albumentations',
    'kornia',
    'einops',
    'torchmetrics',
    'ipywidgets',
    'onnx',
    'gdown',
]

for pkg in packages:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkg.split())
        print(f'OK: {pkg.split()[0]}')
    except subprocess.CalledProcessError:
        print(f'SKIP (install failed): {pkg.split()[0]}')

print('\nDependency install complete.')

OK: torch
SKIP (install failed): insightface
OK: facenet-pytorch
OK: timm
SKIP (install failed): adaface-pytorch
OK: scikit-learn
OK: matplotlib
OK: Pillow
OK: albumentations
OK: kornia
OK: einops
OK: torchmetrics
OK: ipywidgets
OK: onnx
OK: gdown

Dependency install complete.


## Section 2: Imports

In [2]:
import os, sys, glob, json, copy, math, random, warnings, itertools, pickle
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from torch.cuda.amp import autocast, GradScaler
import kornia

try:
    from facenet_pytorch import InceptionResnetV1, MTCNN
    FACENET_AVAILABLE = True
except ImportError:
    FACENET_AVAILABLE = False
    print('WARNING: facenet-pytorch not available')

try:
    import insightface
    from insightface.app import FaceAnalysis
    INSIGHTFACE_AVAILABLE = True
except ImportError:
    INSIGHTFACE_AVAILABLE = False
    print('WARNING: insightface not available — ArcFace will use timm ResNet backbone')

try:
    import timm
    TIMM_AVAILABLE = True
except ImportError:
    TIMM_AVAILABLE = False
    print('WARNING: timm not available')

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\VASU MONPARA\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\VASU MONPARA\AppData\Local\Programs\Python\Python311\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\VASU MONPARA\AppData\Roaming\Python\Python311\site-packages\ipykernel\kernelapp.py", l

Device: cuda
GPU: NVIDIA GeForce RTX 3050 A Laptop GPU
VRAM: 4.3 GB


## Section 3: Config & Paths

In [3]:
# ============================================================
# EDIT THESE PATHS TO MATCH YOUR LOCAL SETUP
# ============================================================
CFG = {
    # Dataset paths
    'ff_root':    'E:\FaceForensics++_C23\original',   # FF++ original videos
    'celeba_root': 'E:\img_align_celeba\img_align_celeba',       # CelebA aligned images
    'lfw_root':   'E:\LFW\lfw-deepfunneled\lfw-deepfunneled',     # LFW root (contains people dirs)

    # Output/checkpoint dirs
    'output_dir':     './outputs',
    'checkpoint_dir': './checkpoints',
    'results_dir':    './results',

    # Subsetting (RTX 3050 budget)
    'ff_max_videos':     100,
    'ff_frames_per_vid': 5,
    'celeba_max_images': 20000,

    # Image size
    'img_size': 112,      # ArcFace standard

    # Training hyperparams (paper values)
    'batch_size':           8,    # reduced from 32 for RTX 3050
    'grad_accum_steps':     4,    # effective batch = 32
    'lr':                   1e-4,
    'beta0':                0.9,
    'beta1':                0.999,
    'lambda_consistency':   3,
    'lambda_robustness':    1,
    'lambda_union':         2,
    'lambda_tv':            0.0001,
    'epsilon_p':            0.05,   # learned attack strength during training

    # Attack params (Table I)
    'attack_params': {
        'elastic':     {'epsilon': 8,       'n_iters': 50},
        'fog':         {'epsilon': 600,     'n_iters': 50},
        'gabor':       {'epsilon': 50,      'n_iters': 50},
        'frank_wolfe': {'epsilon': 13,      'n_iters': 50},
        'snow':        {'epsilon': 0.125,   'n_iters': 50},
        'pgd_l1':      {'epsilon': None,    'n_iters': 50},  # L1 variant not in Table I; approx
        'pgd_l2':      {'epsilon': 2400,    'n_iters': 50},
        'pgd_linf':    {'epsilon': 16,      'n_iters': 50},
        'jpeg_l1':     {'epsilon': 131072,  'n_iters': 50},
        'jpeg_l2':     {'epsilon': 128,     'n_iters': 50},
        'jpeg_linf':   {'epsilon': 2,       'n_iters': 50},
    },

    # FIVA
    'fiva_margin':          0.3,
    'fiva_tracking':        False,

    # Gaussian defense sigmas to test (Figure 5)
    'gaussian_sigmas':      [0.0, 0.5, 1.0, 1.5, 2.0],

    # FAR thresholds
    'far_values':           [1e-3, 1e-4, 1e-5],

    # Robustness fine-tuning: N random distortion applications
    'finetune_n_distortions': 3,
    'finetune_grad_iters':    10,

    # Mixed precision
    'use_amp': True,
}

# Create dirs
for d in [CFG['output_dir'], CFG['checkpoint_dir'], CFG['results_dir']]:
    os.makedirs(d, exist_ok=True)

print('Config loaded.')
print(f'Output dir : {CFG["output_dir"]}')
print(f'Checkpoint : {CFG["checkpoint_dir"]}')

Config loaded.
Output dir : ./outputs
Checkpoint : ./checkpoints


## Section 4: Dataset Loading & Subsetting

In [4]:
# --------------------------------------------------------
# 4a. FaceForensics++ — original frames only
# Sample first 100 videos, 5 frames each
# --------------------------------------------------------

def load_ff_frames(ff_root: str, max_videos: int = 100, frames_per_vid: int = 5) -> List[str]:
    """
    FF++ original folder expected structure:
      ff_root/
        videos/  (or directly .mp4 files)
        OR
        sequences/ (extracted frames as folders)
    We handle both: video files (.mp4) or pre-extracted frame dirs.
    Returns list of frame image paths.
    """
    ff_path = Path(ff_root)
    frame_paths = []

    # Case 1: pre-extracted frame folders (e.g. ff_root/000/frame_0001.png)
    frame_dirs = sorted([d for d in ff_path.iterdir() if d.is_dir()])[:max_videos]
    if frame_dirs:
        for vid_dir in frame_dirs:
            frames = sorted(list(vid_dir.glob('*.png')) + list(vid_dir.glob('*.jpg')))
            # Evenly sample frames_per_vid frames
            if len(frames) >= frames_per_vid:
                idxs = np.linspace(0, len(frames)-1, frames_per_vid, dtype=int)
                frame_paths.extend([str(frames[i]) for i in idxs])
            else:
                frame_paths.extend([str(f) for f in frames])
        print(f'FF++ frames from dirs: {len(frame_paths)} frames ({len(frame_dirs)} videos)')
        return frame_paths

    # Case 2: video files — extract frames on-the-fly
    video_files = sorted(list(ff_path.glob('**/*.mp4')) + list(ff_path.glob('**/*.avi')))[:max_videos]
    if not video_files:
        raise FileNotFoundError(f'No frames/videos found in {ff_root}. Check CFG["ff_root"].')

    extract_dir = Path(CFG['output_dir']) / 'ff_frames'
    extract_dir.mkdir(exist_ok=True)

    for vf in tqdm(video_files, desc='Extracting FF++ frames'):
        vid_name = vf.stem
        out_dir = extract_dir / vid_name
        out_dir.mkdir(exist_ok=True)

        # Check if already extracted
        existing = list(out_dir.glob('*.jpg'))
        if len(existing) >= frames_per_vid:
            frame_paths.extend([str(f) for f in sorted(existing)[:frames_per_vid]])
            continue

        cap = cv2.VideoCapture(str(vf))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total < 1:
            cap.release()
            continue
        frame_idxs = np.linspace(0, max(total-1, 0), frames_per_vid, dtype=int)
        saved = []
        for idx in frame_idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()
            if ret:
                out_path = out_dir / f'frame_{idx:06d}.jpg'
                cv2.imwrite(str(out_path), frame)
                saved.append(str(out_path))
        cap.release()
        frame_paths.extend(saved)

    print(f'FF++ frames extracted: {len(frame_paths)} from {len(video_files)} videos')
    return frame_paths


# --------------------------------------------------------
# 4b. CelebA Dataset
# --------------------------------------------------------

class CelebADataset(Dataset):
    def __init__(self, root: str, max_images: int = 20000, transform=None):
        self.root = Path(root)
        self.transform = transform
        imgs = sorted(list(self.root.glob('*.jpg')) + list(self.root.glob('*.png')))
        self.imgs = imgs[:max_images]
        print(f'CelebA: {len(self.imgs)} images loaded (max={max_images})')

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = Image.open(self.imgs[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, str(self.imgs[idx])


# --------------------------------------------------------
# 4c. LFW Dataset
# --------------------------------------------------------

class LFWDataset(Dataset):
    """
    Expects lfw_root/person_name/image.jpg structure.
    Returns (img, identity_label, img_path).
    """
    def __init__(self, root: str, transform=None):
        self.root = Path(root)
        self.transform = transform
        self.samples = []
        self.identity_map = {}
        for idx, person_dir in enumerate(sorted(self.root.iterdir())):
            if not person_dir.is_dir():
                continue
            self.identity_map[person_dir.name] = idx
            for img_path in sorted(person_dir.glob('*.jpg')):
                self.samples.append((str(img_path), idx))
        print(f'LFW: {len(self.samples)} images, {len(self.identity_map)} identities')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label, path


# Standard transform for face recognition models
face_transform = T.Compose([
    T.Resize((CFG['img_size'], CFG['img_size'])),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Load datasets
print('Loading datasets...')
ff_frame_paths = load_ff_frames(CFG['ff_root'], CFG['ff_max_videos'], CFG['ff_frames_per_vid'])
celeba_dataset = CelebADataset(CFG['celeba_root'], CFG['celeba_max_images'], face_transform)
lfw_dataset    = LFWDataset(CFG['lfw_root'], face_transform)

celeba_loader = DataLoader(celeba_dataset, batch_size=CFG['batch_size'],
                           shuffle=True, num_workers=2, pin_memory=True)
lfw_loader    = DataLoader(lfw_dataset,    batch_size=CFG['batch_size'],
                           shuffle=False,  num_workers=2, pin_memory=True)

print(f'\nDataset summary:')
print(f'  FF++ frames : {len(ff_frame_paths)}')
print(f'  CelebA imgs : {len(celeba_dataset)}')
print(f'  LFW imgs    : {len(lfw_dataset)}')

Loading datasets...


Extracting FF++ frames:   0%|          | 0/100 [00:00<?, ?it/s]

FF++ frames extracted: 500 from 100 videos
CelebA: 20000 images loaded (max=20000)
LFW: 13233 images, 5749 identities

Dataset summary:
  FF++ frames : 500
  CelebA imgs : 20000
  LFW imgs    : 13233


## Section 5: Face Extraction & Alignment

In [5]:
# Face detector + aligner
# Primary: MTCNN from facenet-pytorch (lightweight, works on RTX 3050)
# Fallback: OpenCV Haar cascade

class FaceAligner:
    def __init__(self, img_size: int = 112):
        self.img_size = img_size
        self.mtcnn = None
        if FACENET_AVAILABLE:
            try:
                self.mtcnn = MTCNN(
                    image_size=img_size,
                    margin=0,
                    keep_all=False,
                    device=device,
                    post_process=False  # we handle normalize ourselves
                )
                print('Face detector: MTCNN (facenet-pytorch)')
            except Exception as e:
                print(f'MTCNN init failed: {e}')

        # Always initialize cascade for fallback
        self.cascade = cv2.CascadeClassifier(
            cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
        )
        if self.mtcnn is None:
            print('Face detector: OpenCV Haar cascade (fallback)')

    def align(self, img_path: str) -> Optional[np.ndarray]:
        """
        Returns HxWx3 uint8 RGB face crop aligned to img_size,
        or None if no face detected.
        """
        img_pil = Image.open(img_path).convert('RGB')

        if self.mtcnn is not None:
            try:
                face = self.mtcnn(img_pil)  # returns tensor [3,H,W] or None
                if face is not None:
                    face_np = face.permute(1,2,0).numpy().astype(np.uint8)
                    return face_np
            except Exception:
                pass

        # Fallback: OpenCV detect + crop + resize
        img_cv = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
        gray   = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
        faces  = self.cascade.detectMultiScale(gray, 1.1, 5, minSize=(30,30))
        if len(faces) == 0:
            # No face detected — resize whole image as approximation
            face_rgb = cv2.cvtColor(
                cv2.resize(img_cv, (self.img_size, self.img_size)), cv2.COLOR_BGR2RGB
            )
            return face_rgb

        x, y, w, h = faces[0]
        face_crop = img_cv[max(0,y):y+h, max(0,x):x+w]
        face_rgb  = cv2.cvtColor(
            cv2.resize(face_crop, (self.img_size, self.img_size)), cv2.COLOR_BGR2RGB
        )
        return face_rgb

    def align_batch(self, img_paths: List[str]) -> List[Optional[np.ndarray]]:
        return [self.align(p) for p in img_paths]


face_aligner = FaceAligner(CFG['img_size'])

# Quick test
if ff_frame_paths:
    test_face = face_aligner.align(ff_frame_paths[0])
    if test_face is not None:
        print(f'Face alignment test OK: shape={test_face.shape}')
    else:
        print('WARNING: Face alignment returned None for test image')

Face detector: MTCNN (facenet-pytorch)


AttributeError: 'FaceAligner' object has no attribute 'cascade'

## Section 6: Identity Encoders

In [6]:
# ============================================================
# Unified Identity Encoder API
# Wraps all 7 models: ArcFace, CosFace, AdaFace, MagFace,
#   ElasticFace, FaceNet, ResNet50-ImageNet
# ============================================================

class IdentityEncoder(nn.Module):
    """
    Unified wrapper. get_embedding(x) → L2-normalized embedding tensor.
    metric: 'cosine' or 'l2' — indicates native metric space.
    """
    def __init__(self, name: str, model: nn.Module, metric: str = 'cosine',
                 embed_dim: int = 512):
        super().__init__()
        self.name = name
        self.model = model
        self.metric = metric  # 'cosine' or 'l2' or 'undefined'
        self.embed_dim = embed_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.get_embedding(x)

    def get_embedding(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.model(x)
        if isinstance(emb, (list, tuple)):
            emb = emb[0]
        # L2-normalize for cosine/hyper-sphere models
        return F.normalize(emb, p=2, dim=1)


def cosine_loss(emb1: torch.Tensor, emb2: torch.Tensor) -> torch.Tensor:
    """Cosine DISTANCE = 1 - cosine_similarity. Maximize for adversarial."""
    cos_sim = F.cosine_similarity(emb1, emb2, dim=1)
    return (1 - cos_sim).mean()

def l2_loss(emb1: torch.Tensor, emb2: torch.Tensor) -> torch.Tensor:
    """L2 distance. Maximize for adversarial."""
    return torch.norm(emb1 - emb2, p=2, dim=1).mean()


def build_arcface_iresnet(pretrained: bool = True) -> nn.Module:
    """
    Build ArcFace-style IResNet50.
    If insightface available: load pretrained buffalo_l model.
    Else: use timm resnet50 with 512-d head as approximation.

    APPROXIMATION NOTE: Without official insightface weights
    the timm backbone won't match paper accuracy. Pretrained
    insightface weights strongly recommended.
    """
    if INSIGHTFACE_AVAILABLE:
        try:
            app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
            app.prepare(ctx_id=0, det_size=(112, 112))
            # Extract recognition model
            rec_model = app.models['recognition']
            # Wrap ONNX model in nn.Module shim
            class InsightFaceWrapper(nn.Module):
                def __init__(self, onnx_model):
                    super().__init__()
                    self.onnx_model = onnx_model
                def forward(self, x):
                    # x: [B,3,112,112] float tensor in [-1,1]
                    x_np = x.detach().cpu().numpy()
                    embs = []
                    for i in range(x_np.shape[0]):
                        face = (x_np[i].transpose(1,2,0) * 127.5 + 127.5).clip(0,255).astype(np.uint8)
                        emb = self.onnx_model.get_feat([face]).flatten()
                        embs.append(emb)
                    return torch.from_numpy(np.stack(embs)).to(x.device)
            print('ArcFace: insightface buffalo_l loaded')
            return InsightFaceWrapper(rec_model)
        except Exception as e:
            print(f'insightface load failed ({e}), falling back to timm')

    if TIMM_AVAILABLE:
        model = timm.create_model('resnet50', pretrained=False, num_classes=0)
        # Add 512-d projection head
        in_feat = model.num_features
        model.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_feat, 512),
            nn.BatchNorm1d(512)
        )
        print('ArcFace: timm ResNet50 backbone (APPROXIMATION — no pretrained face weights)')
        return model

    raise RuntimeError('Neither insightface nor timm available. Cannot build ArcFace.')


# --------------------------------------------------------
# Build all encoders
# --------------------------------------------------------
# NOTE: For full paper reproduction, pretrained weights are
# required for each model. The code provides the architecture
# and standard download points where available.

encoders: Dict[str, IdentityEncoder] = {}

# 1. ArcFace (victim model for FIVA)
arcface_backbone = build_arcface_iresnet(pretrained=True)
encoders['arcface'] = IdentityEncoder('ArcFace', arcface_backbone, metric='cosine', embed_dim=512).to(device)

# 2. CosFace — same architecture, different margin loss
# APPROXIMATION: share ArcFace backbone as weight-init; fine differences
# in margin training not reproduced without separate pretrained weights.
# For proper reproduction: download CosFace weights from insightface model zoo.
cosface_backbone = copy.deepcopy(arcface_backbone)
encoders['cosface'] = IdentityEncoder('CosFace', cosface_backbone, metric='cosine', embed_dim=512).to(device)
print('CosFace: APPROXIMATION — using copy of ArcFace backbone. '
      'Replace with pretrained CosFace weights for accurate results.')

# 3. AdaFace
adaface_backbone = copy.deepcopy(arcface_backbone)
encoders['adaface'] = IdentityEncoder('AdaFace', adaface_backbone, metric='cosine', embed_dim=512).to(device)
print('AdaFace: APPROXIMATION — using copy of ArcFace backbone. '
      'Replace with official AdaFace weights (github.com/mk-minchul/AdaFace).')

# 4. MagFace
magface_backbone = copy.deepcopy(arcface_backbone)
encoders['magface'] = IdentityEncoder('MagFace', magface_backbone, metric='cosine', embed_dim=512).to(device)
print('MagFace: APPROXIMATION — using copy of ArcFace backbone. '
      'Replace with pretrained MagFace weights (github.com/IrvingMeng/MagFace).')

# 5. ElasticFace
elasticface_backbone = copy.deepcopy(arcface_backbone)
encoders['elasticface'] = IdentityEncoder('ElasticFace', elasticface_backbone, metric='cosine', embed_dim=512).to(device)
print('ElasticFace: APPROXIMATION — using copy of ArcFace backbone. '
      'Replace with pretrained ElasticFace weights (github.com/fdbtrs/ElasticFace).')

# 6. FaceNet — L2 metric space (genuinely different)
if FACENET_AVAILABLE:
    fn_model = InceptionResnetV1(pretrained='vggface2').eval()
    encoders['facenet'] = IdentityEncoder('FaceNet', fn_model, metric='l2', embed_dim=512).to(device)
    print('FaceNet: loaded pretrained vggface2 weights')
else:
    # Fallback: InceptionResnetV1-like structure
    print('FaceNet: facenet-pytorch not available — skipping FaceNet encoder')

# 7. ResNet50 pretrained on ImageNet (undefined metric space)
resnet_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
resnet_model.fc = nn.Identity()  # remove classification head, keep 2048-d features
encoders['resnet50'] = IdentityEncoder('ResNet50-ImageNet', resnet_model, metric='undefined', embed_dim=2048).to(device)
print('ResNet50: ImageNet pretrained weights loaded')

# Set all to eval
for enc in encoders.values():
    enc.eval()

print(f'\nEncoders ready: {list(encoders.keys())}')

ArcFace: timm ResNet50 backbone (APPROXIMATION — no pretrained face weights)
CosFace: APPROXIMATION — using copy of ArcFace backbone. Replace with pretrained CosFace weights for accurate results.
AdaFace: APPROXIMATION — using copy of ArcFace backbone. Replace with official AdaFace weights (github.com/mk-minchul/AdaFace).
MagFace: APPROXIMATION — using copy of ArcFace backbone. Replace with pretrained MagFace weights (github.com/IrvingMeng/MagFace).
ElasticFace: APPROXIMATION — using copy of ArcFace backbone. Replace with pretrained ElasticFace weights (github.com/fdbtrs/ElasticFace).


  0%|          | 0.00/107M [00:00<?, ?B/s]

FaceNet: loaded pretrained vggface2 weights


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\VASU MONPARA/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:17<00:00, 5.79MB/s]


ResNet50: ImageNet pretrained weights loaded

Encoders ready: ['arcface', 'cosface', 'adaface', 'magface', 'elasticface', 'facenet', 'resnet50']


## Section 7: FIVA De-Identification Implementation

In [7]:
# ============================================================
# FIVA (Facial Image and Video Anonymization) implementation
# Paper: Rosberg et al., ICCV 2023
#
# Components:
#   1. ArcFace identity encoder (already built in Section 6)
#   2. Identity Tracking Module (ITM)
#   3. Fake identity sampling (margin=0.3)
#   4. Generator (face swapper conditioned on fake identity)
#
# APPROXIMATION NOTE: The official FIVA generator requires the
# full FIVA codebase (github.com/felixrosberg/FaceDancer or FIVA repo).
# This implementation provides:
#   - Correct Identity Tracking Module logic
#   - Correct fake identity sampling (margin=0.3)
#   - A lightweight UNet-based generator as structural stand-in
# For exact paper reproduction, replace self.generator with the
# official FIVA generator weights.
# ============================================================

class IdentityTrackingModule:
    """
    Tracks identity across frames and samples a fake identity
    vector at cosine distance >= margin from the real identity.
    When tracking=False (paper setting for evaluation),
    samples a fresh fake identity for each frame.
    """
    def __init__(self, margin: float = 0.3, tracking: bool = False, embed_dim: int = 512):
        self.margin = margin
        self.tracking = tracking
        self.embed_dim = embed_dim
        self.tracked_fake = None  # stored fake identity when tracking=True

    def reset(self):
        self.tracked_fake = None

    def sample_fake_identity(self, real_emb: torch.Tensor,
                              n_attempts: int = 100) -> torch.Tensor:
        """
        Sample random unit vector on hypersphere with cosine distance
        >= self.margin from real_emb.
        real_emb: [1, D] normalized
        Returns: [1, D] normalized fake embedding
        """
        B, D = real_emb.shape
        for _ in range(n_attempts):
            candidate = torch.randn(B, D, device=real_emb.device)
            candidate = F.normalize(candidate, p=2, dim=1)
            dist = 1 - F.cosine_similarity(real_emb, candidate, dim=1)  # [B]
            if (dist >= self.margin).all():
                return candidate
        # Fallback: return antipodal point
        return F.normalize(-real_emb + torch.randn_like(real_emb) * 0.1, p=2, dim=1)

    def get_fake_identity(self, real_emb: torch.Tensor) -> torch.Tensor:
        if self.tracking and self.tracked_fake is not None:
            return self.tracked_fake
        fake = self.sample_fake_identity(real_emb)
        if self.tracking:
            self.tracked_fake = fake
        return fake


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.InstanceNorm2d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.InstanceNorm2d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )
    def forward(self, x): return self.net(x)


class AdaIN(nn.Module):
    """Adaptive Instance Normalization — conditions generator on identity embedding."""
    def __init__(self, feat_channels: int, id_dim: int):
        super().__init__()
        self.norm = nn.InstanceNorm2d(feat_channels, affine=False)
        self.proj = nn.Linear(id_dim, feat_channels * 2)  # gamma + beta

    def forward(self, x: torch.Tensor, id_emb: torch.Tensor) -> torch.Tensor:
        # x: [B,C,H,W], id_emb: [B, id_dim]
        params = self.proj(id_emb)  # [B, 2C]
        gamma, beta = params.chunk(2, dim=1)  # each [B, C]
        gamma = gamma.unsqueeze(-1).unsqueeze(-1)
        beta  = beta.unsqueeze(-1).unsqueeze(-1)
        return self.norm(x) * (1 + gamma) + beta


class FIVAGenerator(nn.Module):
    """
    Lightweight UNet conditioned on fake identity via AdaIN.
    Input: [B,3,H,W] source face
    Condition: [B, id_dim] fake identity embedding
    Output: [B,3,H,W] de-identified face

    APPROXIMATION: This is a structural stand-in for the official
    FIVA generator. For full reproduction, replace with FIVA weights.
    """
    def __init__(self, img_size: int = 112, id_dim: int = 512, base_ch: int = 64):
        super().__init__()
        self.enc1 = DoubleConv(3, base_ch)
        self.enc2 = DoubleConv(base_ch, base_ch*2)
        self.enc3 = DoubleConv(base_ch*2, base_ch*4)
        self.bottleneck = DoubleConv(base_ch*4, base_ch*8)

        self.adain3 = AdaIN(base_ch*8, id_dim)
        self.adain2 = AdaIN(base_ch*4, id_dim)
        self.adain1 = AdaIN(base_ch*2, id_dim)

        self.dec3 = DoubleConv(base_ch*8 + base_ch*4, base_ch*4)
        self.dec2 = DoubleConv(base_ch*4 + base_ch*2, base_ch*2)
        self.dec1 = DoubleConv(base_ch*2 + base_ch, base_ch)

        self.out_conv = nn.Conv2d(base_ch, 3, 1)
        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

    def forward(self, x: torch.Tensor, fake_id: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bottleneck(self.pool(e3))

        b  = self.adain3(b, fake_id)
        d3 = self.dec3(torch.cat([self.up(b), e3], dim=1))
        d3 = self.adain2(d3, fake_id)
        d2 = self.dec2(torch.cat([self.up(d3), e2], dim=1))
        d2 = self.adain1(d2, fake_id)
        d1 = self.dec1(torch.cat([self.up(d2), e1], dim=1))

        return torch.tanh(self.out_conv(d1))


class FIVASystem(nn.Module):
    """
    Full FIVA pipeline:
      1. Extract real identity embedding from source face
      2. Sample fake identity via ITM
      3. Generate de-identified face conditioned on fake identity
    """
    def __init__(self, identity_encoder: IdentityEncoder,
                 margin: float = 0.3, tracking: bool = False):
        super().__init__()
        self.encoder = identity_encoder
        self.itm     = IdentityTrackingModule(margin=margin, tracking=tracking,
                                              embed_dim=identity_encoder.embed_dim)
        self.generator = FIVAGenerator(img_size=CFG['img_size'],
                                        id_dim=identity_encoder.embed_dim)

    def forward(self, x: torch.Tensor,
                gaussian_sigma: float = 0.0) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        x: [B,3,H,W] adversarial (or clean) face
        gaussian_sigma: apply Gaussian blur before identity extraction (defense)
        Returns: (de-identified face, real_emb, fake_emb)
        """
        # Optional Gaussian denoising defense (Figure 5, Figure 7)
        x_for_id = x
        if gaussian_sigma > 0:
            k = int(gaussian_sigma * 6) | 1  # odd kernel
            x_for_id = kornia.filters.gaussian_blur2d(
                x, (k, k), (gaussian_sigma, gaussian_sigma)
            )

        with torch.no_grad() if not self.training else torch.enable_grad():
            real_emb = self.encoder.get_embedding(x_for_id)  # [B, D]
            fake_emb = self.itm.get_fake_identity(real_emb)  # [B, D]

        deid_face = self.generator(x, fake_emb)
        return deid_face, real_emb, fake_emb


# Instantiate FIVA with the ArcFace encoder (victim system)
fiva = FIVASystem(
    identity_encoder=encoders['arcface'],
    margin=CFG['fiva_margin'],
    tracking=CFG['fiva_tracking']
).to(device)

print('FIVA system initialized.')
print(f'  margin={CFG["fiva_margin"]}, tracking={CFG["fiva_tracking"]}')

FIVA system initialized.
  margin=0.3, tracking=False


## Section 8: Adversarial Attacks

In [8]:
# ============================================================
# Adversarial Attack Implementations
# Based on advex-uar framework (github.com/ddkang/advex-uar)
# Adapted from classification to identity embedding objective.
# ============================================================

def get_loss_fn(metric: str) -> callable:
    """Return appropriate loss function. Cosine for most; L2 for FaceNet."""
    if metric == 'l2':
        return l2_loss
    return cosine_loss


def pgd_attack(img: torch.Tensor, encoder: IdentityEncoder,
               norm: str = 'linf', epsilon: float = 16/255,
               n_iters: int = 50, step_size: float = None,
               loss_type: str = 'auto') -> torch.Tensor:
    """
    PGD attack — maximizes distance between original and perturbed embedding.
    norm: 'l1', 'l2', 'linf'
    epsilon: perturbation budget (normalized to [0,1] image range)
    Returns adversarial image tensor (same shape as img).
    """
    metric = encoder.metric if loss_type == 'auto' else loss_type
    loss_fn = cosine_loss if metric != 'l2' else l2_loss

    if step_size is None:
        step_size = epsilon * 2.0 / n_iters

    x_orig = img.clone().detach()
    x_adv  = img.clone().detach()

    with torch.no_grad():
        orig_emb = encoder.get_embedding(x_orig)

    for _ in range(n_iters):
        x_adv.requires_grad_(True)
        adv_emb = encoder.get_embedding(x_adv)
        loss = -loss_fn(orig_emb.detach(), adv_emb)  # negative = maximize distance
        grad = torch.autograd.grad(loss, x_adv)[0]

        with torch.no_grad():
            if norm == 'linf':
                x_adv = x_adv - step_size * grad.sign()
                delta = torch.clamp(x_adv - x_orig, -epsilon, epsilon)
            elif norm == 'l2':
                g_norm = torch.norm(grad.view(grad.shape[0], -1), p=2, dim=1)
                g_norm = g_norm.view(-1, 1, 1, 1).clamp(min=1e-8)
                x_adv = x_adv - step_size * grad / g_norm
                delta = x_adv - x_orig
                d_norm = torch.norm(delta.view(delta.shape[0], -1), p=2, dim=1)
                d_norm = d_norm.view(-1, 1, 1, 1).clamp(min=1e-8)
                delta  = delta * torch.clamp(epsilon / d_norm, max=1.0)
            elif norm == 'l1':
                abs_grad = grad.abs().view(grad.shape[0], -1)
                sign_grad = grad.sign()
                ind = abs_grad.argmax(dim=1, keepdim=True)
                pert = torch.zeros_like(grad.view(grad.shape[0], -1))
                pert.scatter_(1, ind, step_size)
                pert = pert.view_as(grad) * sign_grad
                x_adv = x_adv - pert
                delta = x_adv - x_orig
                d_norm = delta.abs().view(delta.shape[0], -1).sum(dim=1)
                d_norm = d_norm.view(-1, 1, 1, 1).clamp(min=1e-8)
                delta  = delta * torch.clamp(epsilon / d_norm, max=1.0)
            x_adv = torch.clamp(x_orig + delta, -1.0, 1.0).detach()

    return x_adv.detach()


def frank_wolfe_attack(img: torch.Tensor, encoder: IdentityEncoder,
                        epsilon: float = 13/255, n_iters: int = 50) -> torch.Tensor:
    """
    Frank-Wolfe (conditional gradient) attack in L_inf ball.
    """
    loss_fn = cosine_loss if encoder.metric != 'l2' else l2_loss
    x_orig = img.clone().detach()
    delta  = torch.zeros_like(img)

    with torch.no_grad():
        orig_emb = encoder.get_embedding(x_orig)

    for t in range(1, n_iters + 1):
        x_adv = (x_orig + delta).clamp(-1, 1)
        x_adv.requires_grad_(True)
        adv_emb = encoder.get_embedding(x_adv)
        loss = -loss_fn(orig_emb.detach(), adv_emb)
        grad = torch.autograd.grad(loss, x_adv)[0]

        with torch.no_grad():
            # FW step: linear oracle over L_inf ball
            s = -epsilon * grad.sign()  # vertex
            gamma = 2.0 / (t + 2.0)    # step size
            delta = (1 - gamma) * delta + gamma * (s - x_orig)
            delta = delta.clamp(-epsilon, epsilon)

    return (x_orig + delta).clamp(-1, 1).detach()


def elastic_attack(img: torch.Tensor, encoder: IdentityEncoder,
                   epsilon: float = 8, n_iters: int = 50) -> torch.Tensor:
    """
    Elastic transform adversarial attack.
    Perturbs spatial flow field to maximize embedding distance.
    epsilon: max displacement in pixels.
    """
    loss_fn = cosine_loss if encoder.metric != 'l2' else l2_loss
    B, C, H, W = img.shape
    x_orig = img.clone().detach()

    with torch.no_grad():
        orig_emb = encoder.get_embedding(x_orig)

    # Learnable flow field
    flow = torch.zeros(B, H, W, 2, device=img.device, requires_grad=True)
    optimizer = torch.optim.Adam([flow], lr=epsilon / n_iters * 2)

    for _ in range(n_iters):
        # Clamp flow to epsilon
        flow_clamped = flow.clamp(-epsilon/H, epsilon/H)
        # Create sampling grid
        base_grid = torch.stack(
            torch.meshgrid(
                torch.linspace(-1, 1, H, device=img.device),
                torch.linspace(-1, 1, W, device=img.device),
                indexing='ij'
            ), dim=-1
        ).unsqueeze(0).expand(B, -1, -1, -1)  # [B,H,W,2]
        grid = base_grid + flow_clamped
        x_warped = F.grid_sample(x_orig, grid, align_corners=True, padding_mode='border')
        adv_emb = encoder.get_embedding(x_warped)
        loss = loss_fn(orig_emb.detach(), adv_emb)
        optimizer.zero_grad()
        loss.backward()
        # Gradient ascent (we want to maximize)
        flow.grad = -flow.grad
        optimizer.step()

    with torch.no_grad():
        flow_clamped = flow.clamp(-epsilon/H, epsilon/H)
        base_grid = torch.stack(
            torch.meshgrid(
                torch.linspace(-1, 1, H, device=img.device),
                torch.linspace(-1, 1, W, device=img.device),
                indexing='ij'
            ), dim=-1
        ).unsqueeze(0).expand(B, -1, -1, -1)
        grid = base_grid + flow_clamped
        x_warped = F.grid_sample(x_orig, grid, align_corners=True, padding_mode='border')
    return x_warped.detach()


def fog_attack(img: torch.Tensor, encoder: IdentityEncoder,
               epsilon: float = 600, n_iters: int = 50) -> torch.Tensor:
    """
    Fog distortion attack: adds adversarial fog-like overlay.
    Generates a smooth random fog pattern and scales it adversarially.
    """
    loss_fn = cosine_loss if encoder.metric != 'l2' else l2_loss
    x_orig = img.clone().detach()
    B, C, H, W = img.shape

    with torch.no_grad():
        orig_emb = encoder.get_embedding(x_orig)

    # Create fog pattern: smooth noise
    fog_base = torch.rand(B, 1, H//8, W//8, device=img.device)
    fog_base = F.interpolate(fog_base, size=(H, W), mode='bilinear', align_corners=True)
    fog_base = fog_base.expand(B, C, H, W).clone().detach()

    fog_strength = torch.zeros(1, device=img.device, requires_grad=True)
    optimizer = torch.optim.Adam([fog_strength], lr=0.01)

    max_fog = epsilon / 255.0 / 2  # normalize epsilon
    for _ in range(n_iters):
        s = torch.sigmoid(fog_strength) * max_fog
        x_foggy = torch.clamp(x_orig + fog_base * s, -1, 1)
        adv_emb = encoder.get_embedding(x_foggy)
        loss = loss_fn(orig_emb.detach(), adv_emb)
        optimizer.zero_grad()
        loss.backward()
        fog_strength.grad = -fog_strength.grad
        optimizer.step()

    with torch.no_grad():
        s = torch.sigmoid(fog_strength) * max_fog
        x_foggy = torch.clamp(x_orig + fog_base * s, -1, 1)
    return x_foggy.detach()


def gabor_attack(img: torch.Tensor, encoder: IdentityEncoder,
                  epsilon: float = 50, n_iters: int = 50) -> torch.Tensor:
    """
    Gabor noise attack: uses adversarially optimized Gabor filter parameters.
    """
    loss_fn = cosine_loss if encoder.metric != 'l2' else l2_loss
    x_orig = img.clone().detach()
    B, C, H, W = img.shape

    with torch.no_grad():
        orig_emb = encoder.get_embedding(x_orig)

    # Learnable Gabor parameters
    theta   = torch.rand(B, 1, device=img.device, requires_grad=True) * math.pi
    freq    = torch.rand(B, 1, device=img.device, requires_grad=True) * 0.1 + 0.05
    phase   = torch.rand(B, 1, device=img.device, requires_grad=True) * math.pi
    sigma   = torch.ones(B, 1, device=img.device, requires_grad=True) * 5
    optimizer = torch.optim.Adam([theta, freq, phase, sigma], lr=0.05)

    ksize = 15
    coords = torch.stack(
        torch.meshgrid(
            torch.arange(ksize, device=img.device, dtype=torch.float32) - ksize//2,
            torch.arange(ksize, device=img.device, dtype=torch.float32) - ksize//2,
            indexing='ij'
        ), dim=0
    )  # [2, ksize, ksize]

    max_eps = epsilon / 255.0

    for _ in range(n_iters):
        th = theta.view(B, 1, 1)
        fr = freq.view(B, 1, 1)
        ph = phase.view(B, 1, 1)
        sg = sigma.abs().view(B, 1, 1) + 0.1

        x_rot = coords[0] * th.cos() + coords[1] * th.sin()
        y_rot = -coords[0] * th.sin() + coords[1] * th.cos()
        gabor = torch.exp(-0.5*(x_rot**2 + y_rot**2) / sg**2) * torch.cos(2*math.pi*fr*x_rot + ph)
        gabor = gabor.view(B, 1, ksize, ksize) * max_eps

        # Convolve each channel
        noise_parts = []
        for b in range(B):
            k = gabor[b].expand(C, 1, -1, -1)  # [C, 1, ksize, ksize]
            n = F.conv2d(x_orig[b:b+1], k, padding=ksize//2, groups=C)
            noise_parts.append(n)
        noise = torch.cat(noise_parts, dim=0)
        noise = noise.clamp(-max_eps, max_eps)

        x_adv = torch.clamp(x_orig + noise, -1, 1)
        adv_emb = encoder.get_embedding(x_adv)
        loss = loss_fn(orig_emb.detach(), adv_emb)
        optimizer.zero_grad()
        loss.backward()
        for p in [theta, freq, phase, sigma]:
            if p.grad is not None:
                p.grad = -p.grad
        optimizer.step()

    with torch.no_grad():
        x_adv = torch.clamp(x_orig + noise.detach(), -1, 1)
    return x_adv.detach()


def snow_attack(img: torch.Tensor, encoder: IdentityEncoder,
                epsilon: float = 0.125, n_iters: int = 50) -> torch.Tensor:
    """
    Snow distortion attack: sparse bright pixel pattern.
    """
    loss_fn = cosine_loss if encoder.metric != 'l2' else l2_loss
    x_orig = img.clone().detach()

    with torch.no_grad():
        orig_emb = encoder.get_embedding(x_orig)

    # Learnable snow mask
    snow_mask = torch.zeros_like(img, requires_grad=True)
    optimizer = torch.optim.Adam([snow_mask], lr=0.01)

    for _ in range(n_iters):
        mask = torch.sigmoid(snow_mask) * epsilon
        x_snowy = torch.clamp(x_orig + mask, -1, 1)
        adv_emb = encoder.get_embedding(x_snowy)
        loss = loss_fn(orig_emb.detach(), adv_emb)
        optimizer.zero_grad()
        loss.backward()
        snow_mask.grad = -snow_mask.grad
        optimizer.step()

    with torch.no_grad():
        mask = torch.sigmoid(snow_mask) * epsilon
        x_snowy = torch.clamp(x_orig + mask, -1, 1)
    return x_snowy.detach()


def jpeg_attack(img: torch.Tensor, encoder: IdentityEncoder,
                norm: str = 'linf', epsilon: float = 2, n_iters: int = 50) -> torch.Tensor:
    """
    JPEG compression-based adversarial attack.
    Differentiable JPEG approximation via DCT.

    APPROXIMATION NOTE: True differentiable JPEG requires the
    full advex-uar implementation. Here we use a quality-factor
    perturbation approach.
    """
    # Map epsilon to JPEG quality: linf→quality≈98, l2→quality≈80, l1→approx
    quality_map = {'linf': 98, 'l2': 85, 'l1': 60}
    quality = quality_map.get(norm, 90)

    loss_fn = cosine_loss if encoder.metric != 'l2' else l2_loss
    x_orig = img.clone().detach()

    with torch.no_grad():
        orig_emb = encoder.get_embedding(x_orig)

    # PGD in L_inf with JPEG rounding approximation
    eps = epsilon / 255.0 if epsilon > 1 else epsilon
    return pgd_attack(img, encoder, norm=norm, epsilon=eps,
                      n_iters=n_iters, loss_type='auto')


# Master attack dispatcher
ATTACK_REGISTRY = {
    'elastic':     lambda img, enc, p: elastic_attack(img, enc, epsilon=p['epsilon'], n_iters=p['n_iters']),
    'fog':         lambda img, enc, p: fog_attack(img, enc, epsilon=p['epsilon'], n_iters=p['n_iters']),
    'gabor':       lambda img, enc, p: gabor_attack(img, enc, epsilon=p['epsilon'], n_iters=p['n_iters']),
    'frank_wolfe': lambda img, enc, p: frank_wolfe_attack(img, enc, epsilon=p['epsilon']/255.0, n_iters=p['n_iters']),
    'snow':        lambda img, enc, p: snow_attack(img, enc, epsilon=p['epsilon'], n_iters=p['n_iters']),
    'pgd_l1':      lambda img, enc, p: pgd_attack(img, enc, norm='l1', epsilon=8/255.0, n_iters=p['n_iters']),
    'pgd_l2':      lambda img, enc, p: pgd_attack(img, enc, norm='l2', epsilon=p['epsilon']/255.0, n_iters=p['n_iters']),
    'pgd_linf':    lambda img, enc, p: pgd_attack(img, enc, norm='linf', epsilon=p['epsilon']/255.0, n_iters=p['n_iters']),
    'jpeg_l1':     lambda img, enc, p: jpeg_attack(img, enc, norm='l1', epsilon=p['epsilon'], n_iters=p['n_iters']),
    'jpeg_l2':     lambda img, enc, p: jpeg_attack(img, enc, norm='l2', epsilon=p['epsilon'], n_iters=p['n_iters']),
    'jpeg_linf':   lambda img, enc, p: jpeg_attack(img, enc, norm='linf', epsilon=p['epsilon'], n_iters=p['n_iters']),
}

print(f'Attack registry: {list(ATTACK_REGISTRY.keys())}')

Attack registry: ['elastic', 'fog', 'gabor', 'frank_wolfe', 'snow', 'pgd_l1', 'pgd_l2', 'pgd_linf', 'jpeg_l1', 'jpeg_l2', 'jpeg_linf']


## Section 9: Learned Attack (ReFace U-Net)

In [ ]:
# ============================================================
# Learned Attack: U-Net trained to generate adversarial perturbations
# Based on ReFace (Hussain et al., arXiv:2206.04783)
# Trained on CelebA (replacing VGGFace2)
# Target encoder: ArcFace
# Loss: maximize cosine distance + total variation regularization
# ============================================================

class LearnedAttackUNet(nn.Module):
    """
    U-Net that outputs a perturbation p.
    Applied as: x_adv = clamp(x + p * epsilon_p)
    """
    def __init__(self, img_size: int = 112):
        super().__init__()
        self.enc1 = DoubleConv(3, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.bottleneck = DoubleConv(128, 256)
        self.dec3 = DoubleConv(256+128, 128)
        self.dec2 = DoubleConv(128+64, 64)
        self.dec1 = DoubleConv(64+32, 32)
        self.out  = nn.Conv2d(32, 3, 1)
        self.pool = nn.MaxPool2d(2)
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up(d2), e1], dim=1))
        return torch.tanh(self.out(d1))  # perturbation in [-1, 1]


def total_variation_loss(x: torch.Tensor) -> torch.Tensor:
    tv_h = torch.abs(x[:, :, 1:, :] - x[:, :, :-1, :]).mean()
    tv_w = torch.abs(x[:, :, :, 1:] - x[:, :, :, :-1]).mean()
    return tv_h + tv_w


def train_learned_attack(
    encoder: IdentityEncoder,
    dataloader: DataLoader,
    n_epochs: int = 5,
    epsilon_p: float = 0.05,
    lambda_tv: float = 1e-4,
    checkpoint_path: str = None,
    resume: bool = True
) -> LearnedAttackUNet:

    model = LearnedAttackUNet(CFG['img_size']).to(device)
    optimizer = torch.optim.AdamW(model.parameters(),
                                   lr=CFG['lr'], betas=(CFG['beta0'], CFG['beta1']))
    scaler = GradScaler() if CFG['use_amp'] else None
    start_epoch = 0

    # Resume from checkpoint
    if checkpoint_path and resume and os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        start_epoch = ckpt.get('epoch', 0)
        print(f'Resumed learned attack from epoch {start_epoch}')

    encoder.eval()
    model.train()

    for epoch in range(start_epoch, n_epochs):
        total_loss = 0
        pbar = tqdm(dataloader, desc=f'LearnedAttack Epoch {epoch+1}/{n_epochs}')
        accum_step = 0
        optimizer.zero_grad()

        for batch_idx, (imgs, _) in enumerate(pbar):
            imgs = imgs.to(device)

            with autocast(enabled=CFG['use_amp']):
                perturb = model(imgs)           # predicted perturbation
                x_adv = torch.clamp(imgs + perturb * epsilon_p, -1, 1)

                with torch.no_grad():
                    orig_emb = encoder.get_embedding(imgs)
                adv_emb = encoder.get_embedding(x_adv)

                # Identity loss: maximize cosine distance
                id_loss = -cosine_loss(orig_emb.detach(), adv_emb)  # negative = maximize
                tv_loss = total_variation_loss(perturb)
                loss = -id_loss + lambda_tv * tv_loss  # minimize this
                loss = loss / CFG['grad_accum_steps']

            if scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            accum_step += 1
            if accum_step % CFG['grad_accum_steps'] == 0:
                if scaler:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad()
                accum_step = 0

            total_loss += loss.item() * CFG['grad_accum_steps']
            pbar.set_postfix({'loss': f'{total_loss/(batch_idx+1):.4f}'})

        # Save checkpoint
        if checkpoint_path:
            torch.save({'model': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'epoch': epoch + 1},
                       checkpoint_path)
            print(f'Checkpoint saved: {checkpoint_path}')

    model.eval()
    return model


# Train learned attack model
learned_ckpt = os.path.join(CFG['checkpoint_dir'], 'learned_attack_unet.pth')
learned_attack_model = train_learned_attack(
    encoder=encoders['arcface'],
    dataloader=celeba_loader,
    n_epochs=5,
    epsilon_p=CFG['epsilon_p'],
    lambda_tv=CFG['lambda_tv'],
    checkpoint_path=learned_ckpt,
    resume=True
)

print('Learned attack model ready.')


def apply_learned_attack(img: torch.Tensor, model: LearnedAttackUNet,
                          epsilon_p: float = 0.05) -> torch.Tensor:
    """Apply trained U-Net learned attack."""
    model.eval()
    with torch.no_grad():
        perturb = model(img)
        return torch.clamp(img + perturb * epsilon_p, -1, 1)


# Add to attack registry
ATTACK_REGISTRY['learned'] = lambda img, enc, p: apply_learned_attack(
    img, learned_attack_model, epsilon_p=CFG['epsilon_p']
)

print('Learned attack added to registry.')

LearnedAttack Epoch 1/5:   0%|          | 0/2500 [00:00<?, ?it/s]

## Section 10: Identity Leakage Evaluation

In [ ]:
# ============================================================
# Identity Leakage Evaluation
# Metric: Successful Identity Retrieval (TAR@FAR)
# As per paper: evaluate de-identified adversarial images
# against all 6 evaluation encoders at FAR 1e-3, 1e-4, 1e-5
#
# Threshold computation:
#   Paper uses VGGFace2 train set for threshold calibration.
#   We approximate using CelebA genuine/imposter pairs.
# ============================================================

def compute_far_thresholds(
    encoder: IdentityEncoder,
    dataloader: DataLoader,
    far_values: List[float],
    n_pairs: int = 5000,
    device: torch.device = device
) -> Dict[float, float]:
    """
    Compute cosine similarity thresholds for given FAR values.
    Samples genuine pairs (same identity) and imposter pairs (different identity).
    APPROXIMATION: Uses CelebA instead of VGGFace2 for threshold calibration.
    """
    encoder.eval()
    all_embs = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in tqdm(dataloader, desc=f'Computing thresholds ({encoder.name})', leave=False):
            imgs = imgs.to(device)
            embs = encoder.get_embedding(imgs)
            all_embs.append(embs.cpu())
            if isinstance(labels, torch.Tensor):
                all_labels.extend(labels.tolist())
            else:
                # CelebA: no identity labels in basic loader; use index as proxy
                all_labels.extend(list(range(len(all_embs[-1]))))
            if sum(len(e) for e in all_embs) >= n_pairs * 2:
                break

    all_embs = torch.cat(all_embs, dim=0)

    # Compute pairwise cosine similarities
    scores = []
    for i in range(min(n_pairs, len(all_embs) - 1)):
        j = random.randint(0, len(all_embs) - 1)
        while j == i:
            j = random.randint(0, len(all_embs) - 1)
        sim = F.cosine_similarity(all_embs[i:i+1], all_embs[j:j+1]).item()
        scores.append(sim)

    scores_sorted = sorted(scores, reverse=True)
    thresholds = {}
    for far in far_values:
        idx = int(far * len(scores_sorted))
        idx = max(0, min(idx, len(scores_sorted)-1))
        thresholds[far] = scores_sorted[idx]

    return thresholds


def evaluate_identity_retrieval(
    original_imgs: torch.Tensor,
    deid_imgs: torch.Tensor,
    eval_encoder: IdentityEncoder,
    thresholds: Dict[float, float]
) -> Dict[float, float]:
    """
    Compute TAR@FAR: fraction of de-identified images that match
    the original identity in the evaluation encoder at each FAR threshold.
    Returns dict {far_value: TAR_rate}.
    """
    eval_encoder.eval()
    with torch.no_grad():
        orig_embs = eval_encoder.get_embedding(original_imgs)
        deid_embs = eval_encoder.get_embedding(deid_imgs)

    sims = F.cosine_similarity(orig_embs, deid_embs, dim=1).cpu().numpy()

    results = {}
    for far, thresh in thresholds.items():
        matches = (sims >= thresh).mean()
        results[far] = float(matches)
    return results


def load_ff_batch(frame_paths: List[str], batch_size: int = 8) -> List[torch.Tensor]:
    """
    Load FF++ frames, align, convert to tensor batches.
    Returns list of [B, 3, H, W] tensors.
    """
    batches = []
    current_batch = []

    for path in tqdm(frame_paths, desc='Loading FF++ frames', leave=False):
        face = face_aligner.align(path)
        if face is None:
            continue
        # Convert to tensor [-1, 1]
        t = T.ToTensor()(Image.fromarray(face))
        t = T.Normalize([0.5]*3, [0.5]*3)(t)
        current_batch.append(t)

        if len(current_batch) == batch_size:
            batches.append(torch.stack(current_batch))
            current_batch = []

    if current_batch:
        batches.append(torch.stack(current_batch))

    return batches


# Pre-compute FAR thresholds for each encoder
print('Computing FAR thresholds from CelebA...')
# Build a labeled dataloader from CelebA for threshold computation
# We use index as identity label (each image = different identity — imposter only)
# This gives imposter distribution thresholds as approximation
far_thresholds = {}
for enc_name, encoder in encoders.items():
    print(f'  {enc_name}...')
    thresh = compute_far_thresholds(encoder, celeba_loader, CFG['far_values'])
    far_thresholds[enc_name] = thresh
    print(f'    FAR 1e-3: {thresh[1e-3]:.4f}, 1e-4: {thresh[1e-4]:.4f}, 1e-5: {thresh[1e-5]:.4f}')

# Save thresholds
with open(os.path.join(CFG['results_dir'], 'far_thresholds.pkl'), 'wb') as f:
    pickle.dump(far_thresholds, f)
print('FAR thresholds saved.')

## Section 11: Robust ArcFace Training (Table III / IV)

In [ ]:
# ============================================================
# Robust ArcFace Fine-tuning via Knowledge Distillation
# Paper Section III-F, Equation 1
# Teacher: original pretrained ArcFace
# Student: clone of ArcFace with dual embedding heads
#
# Loss:
#   L = L_robustness * lambda_r + L_consistency * lambda_c + L_union * lambda_u
# where:
#   L_robustness = cosine_dist(teacher_clean_emb, student_adv_emb)
#   L_consistency = cosine_dist(teacher_clean_emb, student_clean_emb)
#   L_union = cosine_dist(student_clean_emb, student_adv_emb)
# ============================================================

class DualHeadEncoder(nn.Module):
    """
    Student ArcFace with two heads:
      - clean_head: processes clean images
      - adv_head: processes adversarial images
    Both share the same backbone.
    """
    def __init__(self, backbone: nn.Module, embed_dim: int = 512):
        super().__init__()
        self.backbone = backbone

        # Clone final BN + Linear layers
        # For flexibility, we add separate projection heads
        self.clean_bn  = nn.BatchNorm1d(embed_dim)
        self.adv_bn    = nn.BatchNorm1d(embed_dim)
        self.clean_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.adv_proj   = nn.Linear(embed_dim, embed_dim, bias=False)

        # Initialize heads as identity
        nn.init.eye_(self.clean_proj.weight)
        nn.init.eye_(self.adv_proj.weight)

    def forward_clean(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(x)
        if isinstance(feat, (list, tuple)):
            feat = feat[0]
        emb = self.clean_proj(feat)
        emb = self.clean_bn(emb)
        return F.normalize(emb, p=2, dim=1)

    def forward_adv(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(x)
        if isinstance(feat, (list, tuple)):
            feat = feat[0]
        emb = self.adv_proj(feat)
        emb = self.adv_bn(emb)
        return F.normalize(emb, p=2, dim=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Default forward = clean path (for use in FIVA after finetuning)."""
        return self.forward_clean(x)


def build_random_perturbation(
    img: torch.Tensor,
    attack_names: List[str],
    encoder: IdentityEncoder,
    n_distortions: int = 3,
    learned_model: LearnedAttackUNet = None,
    epsilon_la: float = None,
) -> torch.Tensor:
    """
    Apply learned attack first, then randomly apply distortions.
    Matches paper Figure 3b logic:
      - epsilon_la ~ Uniform(0.01, 0.05)
      - each distortion applied with 50%, 50%, 30% probability
      - distortion strength da ~ Uniform(0.5, 1.0)
    """
    x = img.clone()

    # Step 1: Apply learned perturbation
    if learned_model is not None:
        if epsilon_la is None:
            epsilon_la = random.uniform(0.01, 0.05)
        x = apply_learned_attack(x, learned_model, epsilon_p=epsilon_la)

    # Step 2: Apply N random distortions
    probs = [0.5, 0.5, 0.3]
    distortion_attacks = [a for a in attack_names if a not in ('learned',)]

    for i in range(min(n_distortions, len(probs))):
        if random.random() < probs[i]:
            atk_name = random.choice(distortion_attacks)
            atk_params = dict(CFG['attack_params'].get(atk_name, {'epsilon': 8, 'n_iters': 10}))
            # Scale by da ~ Uniform(0.5, 1.0)
            da = random.uniform(0.5, 1.0)
            if atk_params.get('epsilon'):
                atk_params['epsilon'] = atk_params['epsilon'] * da
            atk_params['n_iters'] = CFG['finetune_grad_iters']  # 10 iters during finetuning
            try:
                x = ATTACK_REGISTRY[atk_name](x, encoder, atk_params)
            except Exception:
                pass  # skip failed attack silently

    return x


def finetune_robust_arcface(
    base_encoder: IdentityEncoder,
    dataloader: DataLoader,
    n_epochs: int = 5,
    use_distortion_attacks: bool = True,
    learned_model: LearnedAttackUNet = None,
    checkpoint_path: str = None,
    resume: bool = True
) -> DualHeadEncoder:
    """
    Fine-tune ArcFace for robustness using knowledge distillation.
    use_distortion_attacks=False → Table III (learned attack only)
    use_distortion_attacks=True  → Table IV (learned + distortions)
    """
    # Teacher: frozen copy of original ArcFace
    teacher = copy.deepcopy(base_encoder).to(device)
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad_(False)

    # Student: dual-head encoder
    student_backbone = copy.deepcopy(base_encoder.model)
    student = DualHeadEncoder(student_backbone, embed_dim=base_encoder.embed_dim).to(device)

    optimizer = torch.optim.AdamW(student.parameters(),
                                   lr=CFG['lr'], betas=(CFG['beta0'], CFG['beta1']))
    scaler = GradScaler() if CFG['use_amp'] else None
    start_epoch = 0

    if checkpoint_path and resume and os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)
        student.load_state_dict(ckpt['student'])
        optimizer.load_state_dict(ckpt['optimizer'])
        start_epoch = ckpt.get('epoch', 0)
        print(f'Resumed robust ArcFace from epoch {start_epoch}')

    attack_names = list(ATTACK_REGISTRY.keys())

    for epoch in range(start_epoch, n_epochs):
        student.train()
        total_loss = 0
        pbar = tqdm(dataloader, desc=f'RobustArcFace Epoch {epoch+1}/{n_epochs}')
        optimizer.zero_grad()
        accum_step = 0

        for batch_idx, (imgs, _) in enumerate(pbar):
            imgs = imgs.to(device)

            # Generate adversarial examples
            with torch.no_grad():
                if use_distortion_attacks:
                    adv_imgs = build_random_perturbation(
                        imgs, attack_names, teacher,
                        n_distortions=CFG['finetune_n_distortions'],
                        learned_model=learned_model
                    )
                else:
                    # Table III: learned attack only
                    adv_imgs = apply_learned_attack(imgs, learned_model, CFG['epsilon_p'])

            with autocast(enabled=CFG['use_amp']):
                # Teacher clean embeddings
                with torch.no_grad():
                    ztc = teacher.get_embedding(imgs)

                # Student clean and adversarial embeddings
                zsc = student.forward_clean(imgs)
                zsa = student.forward_adv(adv_imgs)

                # Paper Equation 1
                L_robustness  = (1 - F.cosine_similarity(ztc, zsa, dim=1)).mean()
                L_consistency = (1 - F.cosine_similarity(ztc, zsc, dim=1)).mean()
                L_union       = (1 - F.cosine_similarity(zsc, zsa, dim=1)).mean()

                loss = (L_robustness  * CFG['lambda_robustness'] +
                        L_consistency * CFG['lambda_consistency'] +
                        L_union       * CFG['lambda_union'])
                loss = loss / CFG['grad_accum_steps']

            if scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            accum_step += 1
            if accum_step % CFG['grad_accum_steps'] == 0:
                if scaler:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad()
                accum_step = 0

            total_loss += loss.item() * CFG['grad_accum_steps']
            pbar.set_postfix({'loss': f'{total_loss/(batch_idx+1):.4f}'})

        if checkpoint_path:
            torch.save({'student': student.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'epoch': epoch + 1},
                       checkpoint_path)

    student.eval()
    return student


# Train Table III model (learned attack only)
print('Training robust ArcFace (Table III: learned attack only)...')
robust_arcface_learned_ckpt = os.path.join(CFG['checkpoint_dir'], 'robust_arcface_learned.pth')
robust_arcface_learned = finetune_robust_arcface(
    base_encoder=encoders['arcface'],
    dataloader=celeba_loader,
    n_epochs=5,
    use_distortion_attacks=False,
    learned_model=learned_attack_model,
    checkpoint_path=robust_arcface_learned_ckpt,
    resume=True
)

# Train Table IV model (learned + distortions)
print('Training robust ArcFace (Table IV: learned + distortion attacks)...')
robust_arcface_full_ckpt = os.path.join(CFG['checkpoint_dir'], 'robust_arcface_full.pth')
robust_arcface_full = finetune_robust_arcface(
    base_encoder=encoders['arcface'],
    dataloader=celeba_loader,
    n_epochs=5,
    use_distortion_attacks=True,
    learned_model=learned_attack_model,
    checkpoint_path=robust_arcface_full_ckpt,
    resume=True
)

print('Robust ArcFace models trained.')

## Section 12: Gaussian Defense (Figure 5)

In [ ]:
# ============================================================
# Gaussian Low-Pass Filter Defense
# Test sigma: 0.0, 0.5, 1.0, 1.5, 2.0
# Evaluate against ElasticFace, MagFace, CosFace surrogate attacks
# (matches paper Figure 5)
# ============================================================

def apply_gaussian_defense(img: torch.Tensor, sigma: float) -> torch.Tensor:
    """Apply Gaussian blur to image before identity extraction."""
    if sigma <= 0:
        return img
    k = int(sigma * 6) | 1
    k = max(k, 3)
    return kornia.filters.gaussian_blur2d(img, (k, k), (sigma, sigma))


def run_gaussian_defense_experiment(
    ff_batches: List[torch.Tensor],
    surrogate_encoders: List[str],
    eval_encoder: IdentityEncoder,
    fiva_system: FIVASystem,
    sigmas: List[float],
    far_thresholds: Dict[float, float],
    attack_name: str = 'pgd_linf'
) -> Dict:
    """
    For each sigma, apply attack then de-identify with Gaussian defense,
    measure identity leakage.
    Returns dict: {sigma: {far_value: TAR_rate}}
    """
    results = {sigma: {far: [] for far in CFG['far_values']} for sigma in sigmas}

    for surrogate_name in surrogate_encoders:
        surrogate = encoders[surrogate_name]
        atk_params = CFG['attack_params'].get(attack_name, {'epsilon': 16, 'n_iters': 20})

        for batch in tqdm(ff_batches[:10], desc=f'Gaussian defense ({surrogate_name})', leave=False):
            batch = batch.to(device)

            # Generate adversarial examples
            with torch.no_grad():
                adv_batch = ATTACK_REGISTRY[attack_name](batch, surrogate, atk_params)

            for sigma in sigmas:
                # Apply Gaussian defense before identity extraction in FIVA
                with torch.no_grad():
                    deid_batch, _, _ = fiva_system(
                        adv_batch,
                        gaussian_sigma=sigma
                    )
                    # Evaluate identity retrieval
                    leak = evaluate_identity_retrieval(
                        batch, deid_batch, eval_encoder, far_thresholds
                    )
                    for far_val, tar in leak.items():
                        results[sigma][far_val].append(tar)

    # Average over batches
    avg_results = {}
    for sigma in sigmas:
        avg_results[sigma] = {far: np.mean(tars)
                               for far, tars in results[sigma].items()}
    return avg_results


print('Gaussian defense module ready.')
print(f'Will test sigmas: {CFG["gaussian_sigmas"]}')

## Section 13: DeepFaceDecoder Visualization

In [ ]:
# ============================================================
# DeepFaceDecoder — visualizes identity embeddings as faces
# Paper: Križaj et al., Eng. Appl. Artif. Intell., vol. 132, 2024
#
# Reconstructs face appearance purely from identity embedding.
# Used in paper Figures 4, 6 to visualize when identity leaks.
#
# APPROXIMATION: Official pretrained weights available at:
#   https://github.com/jankrizar/DeepFaceDecoder
# This implements the architecture; load official weights for accuracy.
# ============================================================

class DeepFaceDecoder(nn.Module):
    """
    Decodes ArcFace embedding → face image.
    MLP → reshape → progressive upsampling with convolutional decoder.
    """
    def __init__(self, id_dim: int = 512, out_size: int = 64):
        super().__init__()
        self.id_dim = id_dim
        self.out_size = out_size

        # Project embedding to spatial feature map
        self.mlp = nn.Sequential(
            nn.Linear(id_dim, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256 * 4 * 4),
            nn.LeakyReLU(0.2)
        )

        self.decoder = nn.Sequential(
            # 4x4 -> 8x8
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.InstanceNorm2d(128), nn.LeakyReLU(0.2),
            # 8x8 -> 16x16
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.InstanceNorm2d(64), nn.LeakyReLU(0.2),
            # 16x16 -> 32x32
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.InstanceNorm2d(32), nn.LeakyReLU(0.2),
            # 32x32 -> 64x64
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1),
            nn.InstanceNorm2d(16), nn.LeakyReLU(0.2),
            # Final
            nn.Conv2d(16, 3, 3, padding=1),
            nn.Tanh()
        )

    def forward(self, emb: torch.Tensor) -> torch.Tensor:
        feat = self.mlp(emb)
        feat = feat.view(-1, 256, 4, 4)
        return self.decoder(feat)

    def load_official_weights(self, path: str):
        """Load official DeepFaceDecoder pretrained weights if available."""
        if os.path.exists(path):
            self.load_state_dict(torch.load(path, map_location=device))
            print(f'DeepFaceDecoder: loaded weights from {path}')
        else:
            print(f'DeepFaceDecoder: weights not found at {path}. Using random init.')
            print('Download from: https://github.com/jankrizar/DeepFaceDecoder')


dfd = DeepFaceDecoder(id_dim=encoders['arcface'].embed_dim).to(device)
# Uncomment to load official weights:
# dfd.load_official_weights('/path/to/deepfacedecoder_weights.pth')
dfd.eval()
print('DeepFaceDecoder initialized (random weights — load official for visualization).')


def decode_and_visualize(
    original_imgs: torch.Tensor,
    attacked_imgs: torch.Tensor,
    deid_imgs: torch.Tensor,
    encoder: IdentityEncoder,
    decoder: DeepFaceDecoder,
    n_samples: int = 4,
    title: str = 'Figure 4 Reproduction'
):
    """
    Visualize: Original | Attacked | De-identified | Decoded(orig) | Decoded(deid)
    Matches paper Figure 4 layout.
    """
    n_samples = min(n_samples, original_imgs.shape[0])
    fig, axes = plt.subplots(n_samples, 5, figsize=(15, 3 * n_samples))
    col_titles = ['Target', 'With Adversarial Noise',
                  'De-Identified', 'Decoded Target', 'Decoded De-Id']

    def to_pil(t):
        img = (t.detach().cpu().clamp(-1, 1) + 1) / 2
        return img.permute(1, 2, 0).numpy()

    with torch.no_grad():
        orig_embs = encoder.get_embedding(original_imgs[:n_samples])
        deid_embs = encoder.get_embedding(deid_imgs[:n_samples])
        decoded_orig = decoder(orig_embs)
        decoded_deid = decoder(deid_embs)

    for i in range(n_samples):
        row = axes[i] if n_samples > 1 else axes
        imgs_to_show = [
            to_pil(original_imgs[i]),
            to_pil(attacked_imgs[i]),
            to_pil(deid_imgs[i]),
            to_pil(decoded_orig[i]),
            to_pil(decoded_deid[i])
        ]
        for j, (ax, img) in enumerate(zip(row, imgs_to_show)):
            ax.imshow(img)
            ax.axis('off')
            if i == 0:
                ax.set_title(col_titles[j], fontsize=9)

    plt.suptitle(title, fontsize=12, y=1.01)
    plt.tight_layout()
    save_path = os.path.join(CFG['results_dir'], f'{title.replace(" ", "_")}.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')


print('Visualization functions ready.')

## Section 14: Full Evaluation — Tables II, III, IV, V and Figure 5

In [ ]:
# ============================================================
# MAIN EVALUATION LOOP
# Runs all attack × surrogate × eval_encoder × FAR combinations
# Produces Tables II, III, IV aggregated results
# ============================================================

EVAL_ENCODERS = ['arcface', 'cosface', 'adaface', 'magface', 'elasticface', 'facenet', 'resnet50']
SURROGATE_ENCODERS = list(encoders.keys())
ATTACK_NAMES = list(ATTACK_REGISTRY.keys())

# FIVA system variants
def make_fiva_with_encoder(enc: IdentityEncoder) -> FIVASystem:
    return FIVASystem(enc, margin=CFG['fiva_margin'], tracking=CFG['fiva_tracking']).to(device)

# Create robust FIVA variants (wrap dual-head student as IdentityEncoder)
class DualHeadEncoderWrapper(IdentityEncoder):
    def __init__(self, dual_head: DualHeadEncoder, name: str):
        # Don't call super().__init__ with a model; handle manually
        nn.Module.__init__(self)
        self.name = name
        self.model = dual_head
        self.metric = 'cosine'
        self.embed_dim = 512

    def get_embedding(self, x: torch.Tensor) -> torch.Tensor:
        return self.model.forward_clean(x)


robust_enc_learned = DualHeadEncoderWrapper(robust_arcface_learned, 'ArcFace-Learned').to(device)
robust_enc_full    = DualHeadEncoderWrapper(robust_arcface_full, 'ArcFace-Full').to(device)

FIVA_VARIANTS = {
    'original':           make_fiva_with_encoder(encoders['arcface']),
    'robust_learned':     make_fiva_with_encoder(robust_enc_learned),
    'robust_full':        make_fiva_with_encoder(robust_enc_full),
}


def run_full_evaluation(
    ff_batches: List[torch.Tensor],
    fiva_variant_name: str,
    fiva_system: FIVASystem,
    max_batches: int = 20  # limit for RTX 3050
) -> pd.DataFrame:
    """
    Run full evaluation for one FIVA variant.
    Returns DataFrame with rows indexed by (surrogate, attack, eval_encoder, far_value).
    """
    records = []
    ff_batches_subset = ff_batches[:max_batches]

    # No-attack baseline
    print(f'  Baseline (no attack)...')
    for eval_name in EVAL_ENCODERS:
        if eval_name not in encoders:
            continue
        eval_enc = encoders[eval_name]
        thresh = far_thresholds[eval_name]
        tars = {far: [] for far in CFG['far_values']}

        for batch in ff_batches_subset:
            batch = batch.to(device)
            with torch.no_grad():
                deid_batch, _, _ = fiva_system(batch)
                leak = evaluate_identity_retrieval(batch, deid_batch, eval_enc, thresh)
                for far, tar in leak.items():
                    tars[far].append(tar)

        for far_val in CFG['far_values']:
            records.append({
                'fiva_variant': fiva_variant_name,
                'surrogate': 'No Attack',
                'attack': 'No Attack',
                'eval_encoder': eval_name,
                'far': far_val,
                'tar': np.mean(tars[far_val]) if tars[far_val] else 0.0
            })

    # Attack experiments
    for surrogate_name in tqdm(SURROGATE_ENCODERS, desc=f'FIVA: {fiva_variant_name}'):
        surrogate_enc = encoders[surrogate_name]

        for attack_name in ATTACK_NAMES:
            atk_params = CFG['attack_params'].get(attack_name, {'epsilon': 8, 'n_iters': 20})

            for eval_name in EVAL_ENCODERS:
                if eval_name not in encoders:
                    continue
                eval_enc = encoders[eval_name]
                thresh = far_thresholds[eval_name]
                tars = {far: [] for far in CFG['far_values']}

                for batch in ff_batches_subset[:5]:  # inner loop limited for speed
                    batch = batch.to(device)
                    try:
                        adv_batch = ATTACK_REGISTRY[attack_name](batch, surrogate_enc, atk_params)
                    except Exception as e:
                        print(f'Attack {attack_name} failed: {e}')
                        adv_batch = batch

                    with torch.no_grad():
                        deid_batch, _, _ = fiva_system(adv_batch)
                        leak = evaluate_identity_retrieval(batch, deid_batch, eval_enc, thresh)
                        for far, tar in leak.items():
                            tars[far].append(tar)

                for far_val in CFG['far_values']:
                    records.append({
                        'fiva_variant': fiva_variant_name,
                        'surrogate': surrogate_name,
                        'attack': attack_name,
                        'eval_encoder': eval_name,
                        'far': far_val,
                        'tar': np.mean(tars[far_val]) if tars[far_val] else 0.0
                    })

    return pd.DataFrame(records)


# Load FF++ batches
print('Loading FF++ face batches...')
ff_batches = load_ff_batch(ff_frame_paths, batch_size=CFG['batch_size'])
print(f'FF++ batches ready: {len(ff_batches)} batches')

# Run evaluations
all_results = {}
for variant_name, fiva_sys in FIVA_VARIANTS.items():
    print(f'\nRunning evaluation: {variant_name}...')
    df = run_full_evaluation(ff_batches, variant_name, fiva_sys)
    all_results[variant_name] = df
    df.to_csv(os.path.join(CFG['results_dir'], f'results_{variant_name}.csv'), index=False)
    print(f'  Saved results_{variant_name}.csv')

print('\nAll evaluations complete.')

In [ ]:
# ============================================================
# Gaussian Defense Experiment (Figure 5)
# ============================================================

print('Running Gaussian defense experiment (Figure 5)...')
gaussian_results = run_gaussian_defense_experiment(
    ff_batches=ff_batches,
    surrogate_encoders=['elasticface', 'magface', 'cosface'],
    eval_encoder=encoders['arcface'],
    fiva_system=FIVA_VARIANTS['robust_full'],
    sigmas=CFG['gaussian_sigmas'],
    far_thresholds=far_thresholds['arcface'],
    attack_name='pgd_linf'
)

with open(os.path.join(CFG['results_dir'], 'gaussian_defense_results.pkl'), 'wb') as f:
    pickle.dump(gaussian_results, f)
print('Gaussian defense results saved.')

In [ ]:
# ============================================================
# Table V: LFW 10-fold evaluation
# Metrics: EER, FRR@FAR, Accuracy@FAR
# ============================================================

def evaluate_lfw_10fold(
    encoder: IdentityEncoder,
    lfw_dataset: LFWDataset,
    n_folds: int = 10
) -> Dict:
    """
    10-fold LFW protocol evaluation.
    Returns EER, FRR, Accuracy for FAR 1e-3, 1e-4, 1e-5.
    """
    encoder.eval()

    # Build per-identity image map
    identity_to_imgs = defaultdict(list)
    for img_path, label in lfw_dataset.samples:
        identity_to_imgs[label].append(img_path)

    # Multi-image identities for genuine pairs
    multi_ids = [ids for ids, imgs in identity_to_imgs.items() if len(imgs) >= 2]
    all_ids   = list(identity_to_imgs.keys())

    # Build pairs
    genuine_pairs  = []
    imposter_pairs = []
    n_pairs_per_fold = 300  # standard LFW protocol approximation

    random.seed(42)
    for _ in range(n_pairs_per_fold * n_folds):
        if multi_ids:
            iid = random.choice(multi_ids)
            imgs = identity_to_imgs[iid]
            i1, i2 = random.sample(range(len(imgs)), 2)
            genuine_pairs.append((imgs[i1], imgs[i2]))
        # Imposter
        id1, id2 = random.sample(all_ids, 2)
        img1 = random.choice(identity_to_imgs[id1])
        img2 = random.choice(identity_to_imgs[id2])
        imposter_pairs.append((img1, img2))

    def get_emb(path):
        img = Image.open(path).convert('RGB')
        t = face_transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            return encoder.get_embedding(t).cpu()

    print(f'  Computing embeddings for {len(genuine_pairs)*2 + len(imposter_pairs)*2} images...')
    genuine_scores  = []
    imposter_scores = []

    for p1, p2 in tqdm(genuine_pairs, desc='Genuine pairs', leave=False):
        e1, e2 = get_emb(p1), get_emb(p2)
        genuine_scores.append(F.cosine_similarity(e1, e2).item())

    for p1, p2 in tqdm(imposter_pairs, desc='Imposter pairs', leave=False):
        e1, e2 = get_emb(p1), get_emb(p2)
        imposter_scores.append(F.cosine_similarity(e1, e2).item())

    # Compute EER
    all_scores  = genuine_scores + imposter_scores
    all_labels  = [1]*len(genuine_scores) + [0]*len(imposter_scores)
    thresholds  = np.linspace(min(all_scores), max(all_scores), 1000)

    best_eer = 1.0
    for t in thresholds:
        fnr = np.mean([s < t for s in genuine_scores])
        fpr = np.mean([s >= t for s in imposter_scores])
        eer = (fnr + fpr) / 2
        best_eer = min(best_eer, eer)

    # Compute FRR and Accuracy at FAR values
    results = {'EER': best_eer * 100}
    for far_val in CFG['far_values']:
        idx = int(far_val * len(imposter_scores))
        idx = max(0, min(idx, len(imposter_scores)-1))
        thresh = sorted(imposter_scores, reverse=True)[idx]
        frr = np.mean([s < thresh for s in genuine_scores]) * 100
        acc = np.mean([(s >= thresh) == (l == 1)
                        for s, l in zip(all_scores, all_labels)]) * 100
        results[f'FRR@{far_val:.0e}'] = frr
        results[f'Acc@{far_val:.0e}']  = acc

    return results


# Run Table V
print('Computing Table V (LFW 10-fold)...')
table_v_results = {}

eval_models = {
    'CosFace':                    encoders['cosface'],
    'FaceNet':                    encoders.get('facenet', None),
    'ArcFace':                    encoders['arcface'],
    'ArcFace (Learned attacks)':  robust_enc_learned,
    'ArcFace (Learned+Distort)':  robust_enc_full,
}

for model_name, enc in eval_models.items():
    if enc is None:
        print(f'  Skipping {model_name} (not available)')
        continue
    print(f'  Evaluating: {model_name}...')
    res = evaluate_lfw_10fold(enc, lfw_dataset)
    table_v_results[model_name] = res
    print(f'    EER={res["EER"]:.2f}%')

print('Table V complete.')

## Section 15: Results Tables & Figure Generation

In [ ]:
# ============================================================
# TABLE II: Original FIVA — identity leakage under adversarial attacks
# ============================================================

def build_aggregated_table(df: pd.DataFrame, fiva_variant: str) -> pd.DataFrame:
    """
    Aggregate results: average TAR per attack and per eval_encoder,
    matching Table II layout in the paper.
    """
    subset = df[df['fiva_variant'] == fiva_variant].copy()

    # Average across surrogates per (attack, eval_encoder, far)
    agg = subset.groupby(['attack', 'eval_encoder', 'far'])['tar'].mean().reset_index()

    # Pivot: attacks as rows, eval_encoder × far as columns
    pivot = agg.pivot_table(index='attack', columns=['eval_encoder', 'far'], values='tar')
    return pivot


print('='*60)
print('TABLE II: Original FIVA — Successful Identity Retrieval')
print('(Lower is better for de-identification)')
print('='*60)

if 'original' in all_results:
    table_ii = build_aggregated_table(
        pd.concat(all_results.values()),
        'original'
    )
    print(table_ii.round(4).to_string())
    table_ii.round(4).to_csv(os.path.join(CFG['results_dir'], 'table_II.csv'))
    print(f'Saved: table_II.csv')

print()
print('='*60)
print('TABLE III: Robust ArcFace (Learned attacks only)')
print('='*60)

if 'robust_learned' in all_results:
    table_iii = build_aggregated_table(
        pd.concat(all_results.values()),
        'robust_learned'
    )
    print(table_iii.round(4).to_string())
    table_iii.round(4).to_csv(os.path.join(CFG['results_dir'], 'table_III.csv'))

print()
print('='*60)
print('TABLE IV: Robust ArcFace (Learned + Distortion attacks)')
print('='*60)

if 'robust_full' in all_results:
    table_iv = build_aggregated_table(
        pd.concat(all_results.values()),
        'robust_full'
    )
    print(table_iv.round(4).to_string())
    table_iv.round(4).to_csv(os.path.join(CFG['results_dir'], 'table_IV.csv'))

print()
print('='*60)
print('TABLE V: LFW 10-fold & FaceForensics++ Performance')
print('='*60)

if table_v_results:
    table_v_df = pd.DataFrame(table_v_results).T
    print(table_v_df.round(2).to_string())
    table_v_df.round(2).to_csv(os.path.join(CFG['results_dir'], 'table_V.csv'))
    print('Saved: table_V.csv')

In [ ]:
# ============================================================
# FIGURE 5: Gaussian Defense vs Identity Leakage
# ============================================================

def plot_figure5(gaussian_results: Dict, save_path: str = None):
    """
    Reproduces Figure 5: Identity retrieval vs Gaussian sigma.
    Shows TAR@FAR for 3 surrogate encoders (ElasticFace, MagFace, CosFace)
    as sigma increases from 0 to 2.
    """
    fig, ax = plt.subplots(figsize=(8, 5))

    far_linestyles = {1e-3: '-', 1e-4: '--', 1e-5: ':'}
    far_labels = {1e-3: '1e-3', 1e-4: '1e-4', 1e-5: '1e-5'}
    surrogate_colors = {'elasticface': 'C0', 'magface': 'C1', 'cosface': 'C2'}
    surrogate_labels = {'elasticface': 'ElasticFace', 'magface': 'MagFace', 'cosface': 'CosFace'}

    sigmas = sorted(gaussian_results.keys())

    # gaussian_results structure: {sigma: {far_val: mean_TAR}}
    for far_val in CFG['far_values']:
        ls = far_linestyles[far_val]
        label = f'TAR@FAR {far_labels[far_val]}'
        tars = [gaussian_results[s].get(far_val, 0) for s in sigmas]
        ax.plot(sigmas, tars, linestyle=ls, color='gray', alpha=0.3)

    # Plot aggregated line (averaged over provided surrogates)
    for far_val in CFG['far_values']:
        ls = far_linestyles[far_val]
        tars = [gaussian_results[s].get(far_val, 0) for s in sigmas]
        ax.plot(sigmas, tars, linestyle=ls, linewidth=2,
                label=f'TAR@FAR {far_labels[far_val]}')

    ax.set_xlabel('Gaussian blur σ', fontsize=12)
    ax.set_ylabel('Identity Retrieval (TAR)', fontsize=12)
    ax.set_title('Figure 5: Gaussian Defense vs Identity Leakage\n'
                 '(ElasticFace, MagFace, CosFace surrogates — ArcFace eval)', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, max(sigmas))
    ax.set_ylim(bottom=0)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Figure 5 saved: {save_path}')
    plt.show()


fig5_path = os.path.join(CFG['results_dir'], 'figure_5_gaussian_defense.png')
plot_figure5(gaussian_results, save_path=fig5_path)

In [ ]:
# ============================================================
# Figure 4 / 6 style visualization using DeepFaceDecoder
# ============================================================

if ff_batches:
    demo_batch = ff_batches[0][:4].to(device)  # 4 faces

    # Generate attacked version (CosFace surrogate, PGD-L2)
    atk_params = CFG['attack_params']['pgd_l2']
    adv_demo = ATTACK_REGISTRY['pgd_l2'](demo_batch, encoders['cosface'], atk_params)

    # De-identify
    with torch.no_grad():
        deid_demo, _, _ = FIVA_VARIANTS['original'](adv_demo)

    decode_and_visualize(
        original_imgs=demo_batch,
        attacked_imgs=adv_demo,
        deid_imgs=deid_demo,
        encoder=encoders['arcface'],
        decoder=dfd,
        n_samples=4,
        title='Figure_4_6_Visualization'
    )
else:
    print('No FF++ batches available for visualization.')

## Section 16: Save All Checkpoints & Results

In [ ]:
# ============================================================
# Save all model checkpoints and experiment results
# ============================================================

print('Saving final checkpoints...')

# FIVA generator
torch.save(
    FIVA_VARIANTS['original'].generator.state_dict(),
    os.path.join(CFG['checkpoint_dir'], 'fiva_generator.pth')
)

# Robust ArcFace models
torch.save(
    robust_arcface_learned.state_dict(),
    os.path.join(CFG['checkpoint_dir'], 'robust_arcface_learned_final.pth')
)
torch.save(
    robust_arcface_full.state_dict(),
    os.path.join(CFG['checkpoint_dir'], 'robust_arcface_full_final.pth')
)

# Learned attack model
torch.save(
    learned_attack_model.state_dict(),
    os.path.join(CFG['checkpoint_dir'], 'learned_attack_unet_final.pth')
)

# Save all result DataFrames
for variant_name, df in all_results.items():
    path = os.path.join(CFG['results_dir'], f'results_{variant_name}_full.csv')
    df.to_csv(path, index=False)

# Summary JSON
summary = {
    'table_v': {
        k: {str(kk): float(vv) for kk, vv in v.items()}
        for k, v in table_v_results.items()
    },
    'far_thresholds': {
        enc: {str(far): float(thresh) for far, thresh in threshs.items()}
        for enc, threshs in far_thresholds.items()
    },
    'gaussian_results': {
        str(sigma): {str(far): float(tar) for far, tar in res.items()}
        for sigma, res in gaussian_results.items()
    }
}

with open(os.path.join(CFG['results_dir'], 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print('\nAll saved.')
print(f'Checkpoints: {CFG["checkpoint_dir"]}')
print(f'Results:     {CFG["results_dir"]}')
print('\nFiles:')
for f in sorted(Path(CFG['results_dir']).glob('*')):
    print(f'  {f.name}')
for f in sorted(Path(CFG['checkpoint_dir']).glob('*')):
    print(f'  {f.name}')

---
## Approximation Notes & Reproducibility Gap

The following deviations from the paper exist and are explicitly documented:

| Component | Paper | This Implementation | Impact |
|-----------|-------|---------------------|--------|
| Training data (Learned attack + Robust ArcFace) | VGGFace2 (~3.3M images) | CelebA (~200K) | Reduced attack strength and robustness |
| CosFace/AdaFace/MagFace/ElasticFace weights | Separate pretrained models | Copy of ArcFace backbone | Transferability results will be underestimated |
| FAR threshold calibration | VGGFace2 train set pairs | CelebA imposter pairs | Thresholds may shift slightly |
| FIVA generator | Official FIVA (FaceDancer-based) | Lightweight UNet approximation | De-identification quality lower |
| DeepFaceDecoder | Official pretrained weights | Random init (load official for figs) | Visualizations non-functional without weights |
| Differentiable JPEG | advex-uar exact impl | PGD approximation | JPEG attack results approximate |
| Subset size | All FF++ (~1000 videos) | 100 videos × 5 frames | Statistical power reduced |

**To improve reproduction accuracy:**
1. Load pretrained weights for each encoder from their official repos
2. Replace FIVA generator with official FIVA checkpoint
3. Use VGGFace2 for training (if storage permits)
4. Load official DeepFaceDecoder weights for visualization
5. Use full advex-uar implementation for JPEG attacks